In [ ]:
# PHASE 1D: portable runtime path bootstrap
import os as _tfm_os
from pathlib import Path as _TFMPath
_TFM_PUBLIC_PROJECT_ROOT = _TFMPath(
    _tfm_os.environ.get("TFM_PROJECT_ROOT", str(_TFMPath.cwd()))
).expanduser().resolve()
_TFM_PUBLIC_COLAB_WORKDIR = _TFMPath(
    _tfm_os.environ.get(
        "TFM_COLAB_WORKDIR",
        "/content" if _TFMPath("/content").is_dir() else str(_TFM_PUBLIC_PROJECT_ROOT),
    )
).expanduser().resolve()


# Phase G — ejecución científica completa en Google Colab (wrapper v1r1)

**Campaña:** `DATA-SCALING-EXPLORATORY-001`  
**Protocolo:** `PHASE-G-PROTOCOL-001`  
**Autorización:** `074`  
**Modo:** `EJECUCIÓN PRODUCTIVA AUTORIZADA`  
**Fecha:** `2026-08-25`

Este notebook **no reejecuta el preflight**. Aplica el gate mínimo previo a ejecución:

```text
runtime GPU exacto
→ bundle exacto
→ runner exacto
→ 074 exacto
→ ausencia de outputs previos
→ --execute --device cuda
```

La matriz científica permanece congelada:

```text
histories = 1 / 2 / 4 / 8 weeks
models = Persistence / VAR(5) / GRU-N06 / Transformer-T01-L24
horizons = H1 / H3 / H6 / H12
test = June Week #4
seed = 20260820
```

No se permite instalar paquetes, retocar hiperparámetros, cambiar modelos, modificar
el bundle ni editar resultados durante esta ejecución.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import subprocess
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import statsmodels
import torch

# CUDA determinista, igual que en el runner congelado.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

WRAPPER_ID = "PHASE-G-DATA-SCALING-FULL-EXECUTION-COLAB-V1R1"
WRAPPER_VERSION = "1.0.1"
DATE_TAG = "2026-08-25"

BUNDLE_NAME = "phase_g_data_scaling_exec_v1_2026-08-25.zip"
BUNDLE_SHA = "2df26912a5dd5e7b4c1a123a85286e3a0dfc73a6b8cd2a7eb3d26fdfb72793da"
BUNDLE_ROOT_NAME = "phase_g_data_scaling_exec_v1_2026-08-25"

BUNDLE_MANIFEST_NAME = "phase_g_data_scaling_exec_bundle_manifest_v1_2026-08-25.json"
BUNDLE_MANIFEST_SHA = "73256d8179d4c4247f05652a32436503514b1c8dae5f491adca1616b65308869"

RUNNER_NAME = "run_phase_g_data_scaling_exploratory_v1_2026-08-25.py"
RUNNER_SHA = "b04f64aea65b7c81cba3d9bf99619c53b24258ca5417f28ef8ad29e976348966"

AUTH074_NAME = "074_phase_g_scientific_runner_preflight_closure_data_scaling_execution_authorization_2026-08-25.md"
AUTH074_SHA = "216bf95aea67ff6084b496f71d4fa088a25e2aaad90392067886af2a151f05f9"

CONTINUITY_MANIFEST_NAME = "phase_g_continuity_series_preparation_v2_2026-08-25_manifest.json"
CONTINUITY_MANIFEST_SHA = "5423845bb70a85331d8b780fec883cfd74c175288872674240fb6054ab689f7c"

EXPECTED_RUNTIME = {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "torch_cuda": "12.8",
    "cudnn": 91900,
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "statsmodels": "0.14.6",
    "gpu_name": "Tesla T4",
}

HARD_STAGE_SECONDS = 1800.0

CONTENT = Path((str(_TFM_PUBLIC_COLAB_WORKDIR)))
UNPACK_PARENT = CONTENT / "phase_g_data_scaling_full_execution_v1r1_unpack"
SCIENTIFIC_OUT = CONTENT / "phase_g_data_scaling_scientific_outputs_v1"
EXEC_CONSOLE = CONTENT / "phase_g_data_scaling_full_execution_console_v1r1_2026-08-25.txt"

WRAPPER_REPORT = CONTENT / "phase_g_data_scaling_full_execution_colab_v1r1_2026-08-25_report.txt"
WRAPPER_MANIFEST = CONTENT / "phase_g_data_scaling_full_execution_colab_v1r1_2026-08-25_manifest.json"
RESULTS_ZIP = CONTENT / "phase_g_data_scaling_results_v1_2026-08-25.zip"

EXPECTED_SCIENTIFIC_FILES = [
    "phase_g_data_scaling_predictions_v1_2026-08-25.parquet",
    "phase_g_data_scaling_metrics_v1_2026-08-25.csv",
    "phase_g_data_scaling_summary_v1_2026-08-25.csv",
    "phase_g_data_scaling_runtime_v1_2026-08-25.csv",
    "phase_g_data_scaling_neural_epoch_registry_v1_2026-08-25.csv",
    "phase_g_data_scaling_cell_status_v1_2026-08-25.csv",
    "phase_g_data_scaling_report_v1_2026-08-25.txt",
    "phase_g_data_scaling_manifest_v1_2026-08-25.json",
]

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def check(name, condition, detail=""):
    ok = bool(condition)
    print(f"{name:88s}: {'PASS' if ok else 'FAIL'}" + (f" | {detail}" if str(detail) else ""))
    if not ok:
        raise RuntimeError(f"{name}: {detail}")

def deterministic_zip(zip_path: Path, members):
    check("ZIP resultados ausente antes de crear", not zip_path.exists(), str(zip_path))
    fixed_time = (2026, 8, 25, 12, 0, 0)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
        for src, arcname in sorted(members, key=lambda x: x[1]):
            info = zipfile.ZipInfo(arcname)
            info.date_time = fixed_time
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o644 << 16
            zf.writestr(info, src.read_bytes())


## 1. Gate del runtime GPU

La ejecución queda autorizada únicamente con el mismo stack software y la misma GPU
observada en el preflight cerrado por `074`.

Si cualquier comprobación falla, **no continúes** y no instales paquetes.


In [ ]:
runtime = {
    "python": sys.version.split()[0],
    "python_full": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "statsmodels": statsmodels.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_vram_bytes": (
        int(torch.cuda.get_device_properties(0).total_memory)
        if torch.cuda.is_available() else None
    ),
    "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
}

print(json.dumps(runtime, indent=2, ensure_ascii=False))

check("Python exacto", runtime["python"] == EXPECTED_RUNTIME["python"], runtime["python"])
check("PyTorch exacto", runtime["torch"] == EXPECTED_RUNTIME["torch"], runtime["torch"])
check("CUDA exacta", runtime["torch_cuda"] == EXPECTED_RUNTIME["torch_cuda"], runtime["torch_cuda"])
check("cuDNN exacto", runtime["cudnn"] == EXPECTED_RUNTIME["cudnn"], runtime["cudnn"])
check("NumPy exacto", runtime["numpy"] == EXPECTED_RUNTIME["numpy"], runtime["numpy"])
check("pandas exacto", runtime["pandas"] == EXPECTED_RUNTIME["pandas"], runtime["pandas"])
check("statsmodels exacto", runtime["statsmodels"] == EXPECTED_RUNTIME["statsmodels"], runtime["statsmodels"])
check("CUDA disponible", runtime["cuda_available"], runtime["gpu_name"])
check("GPU exacta Tesla T4", runtime["gpu_name"] == EXPECTED_RUNTIME["gpu_name"], runtime["gpu_name"])
check("CUBLAS_WORKSPACE_CONFIG exacto", runtime["cublas_workspace_config"] == ":4096:8", runtime["cublas_workspace_config"])

print("PHASE_G_FULL_EXECUTION_RUNTIME_GATE=PASS")


## 2. Subir exactamente el bundle y `074`

Selecciona **exactamente dos archivos**:

```text
phase_g_data_scaling_exec_v1_2026-08-25.zip
074_phase_g_scientific_runner_preflight_closure_data_scaling_execution_authorization_2026-08-25.md
```

La identidad se decide exclusivamente por SHA-256.


In [ ]:
from google.colab import files

uploaded = files.upload()
check("Se han subido exactamente dos ficheros", len(uploaded) == 2, repr(sorted(uploaded.keys())))

by_sha = {}
for received_name, payload in uploaded.items():
    digest = hashlib.sha256(payload).hexdigest()
    print(received_name, digest)
    if digest in by_sha:
        raise RuntimeError(f"SHA duplicado inesperado en upload: {digest}")
    by_sha[digest] = (received_name, payload)

check("Bundle upload exacto", BUNDLE_SHA in by_sha, BUNDLE_SHA)
check("074 upload exacto", AUTH074_SHA in by_sha, AUTH074_SHA)

bundle_path = CONTENT / BUNDLE_NAME
auth074_path = CONTENT / AUTH074_NAME

def canonicalize_uploaded(label, expected_sha, canonical_path):
    received_name, payload = by_sha[expected_sha]
    received_path = CONTENT / received_name

    # google.colab.files.upload() guarda el fichero en $TFM_COLAB_WORKDIR ANTES de devolver.
    # Si el nombre es el canónico, debe existir ya y se acepta solo si el SHA es exacto.
    if canonical_path.exists():
        check(
            f"{label} canónico presente e idéntico",
            sha256(canonical_path) == expected_sha,
            sha256(canonical_path),
        )
    else:
        canonical_path.write_bytes(payload)
        check(
            f"{label} canónico creado con SHA exacto",
            sha256(canonical_path) == expected_sha,
            sha256(canonical_path),
        )

    # Si Colab renombró el upload por colisión, verificamos y eliminamos solo
    # la copia temporal no canónica; nunca se toca el artefacto canónico.
    if received_path != canonical_path and received_path.exists():
        check(
            f"{label} copia temporal idéntica",
            sha256(received_path) == expected_sha,
            sha256(received_path),
        )
        received_path.unlink()
        check(
            f"{label} copia temporal eliminada",
            not received_path.exists(),
            str(received_path),
        )

canonicalize_uploaded("Bundle", BUNDLE_SHA, bundle_path)
canonicalize_uploaded("074", AUTH074_SHA, auth074_path)

check("Bundle SHA final exacto", sha256(bundle_path) == BUNDLE_SHA, sha256(bundle_path))
check("074 SHA final exacto", sha256(auth074_path) == AUTH074_SHA, sha256(auth074_path))

print("PHASE_G_FULL_EXECUTION_UPLOAD_GATE=PASS")


## 3. Auditoría criptográfica mínima antes de `--execute`

No se repite el preflight científico. Solo se verifica que el artefacto de ejecución sea
exactamente el ya auditado y que `074` autorice literalmente la campaña.


In [ ]:
for p in [UNPACK_PARENT, SCIENTIFIC_OUT, EXEC_CONSOLE, WRAPPER_REPORT, WRAPPER_MANIFEST, RESULTS_ZIP]:
    check(f"Ausente antes de ejecución: {p.name}", not p.exists(), str(p))

UNPACK_PARENT.mkdir(parents=False, exist_ok=False)

with zipfile.ZipFile(bundle_path, "r") as zf:
    bad = zf.testzip()
    check("ZIP bundle sin corrupción", bad is None, str(bad))
    zf.extractall(UNPACK_PARENT)

bundle_root = UNPACK_PARENT / BUNDLE_ROOT_NAME
check("Bundle root existe", bundle_root.is_dir(), str(bundle_root))

sha_file = bundle_root / "SHA256SUMS.txt"
check("SHA256SUMS existe", sha_file.is_file(), str(sha_file))

proc_sha = subprocess.run(
    ["sha256sum", "-c", "SHA256SUMS.txt"],
    cwd=bundle_root,
    capture_output=True,
    text=True,
)
print(proc_sha.stdout, end="")
if proc_sha.stderr:
    print(proc_sha.stderr, file=sys.stderr, end="")
check("SHA256SUMS completo", proc_sha.returncode == 0, proc_sha.returncode)

runner = bundle_root / "src/models" / RUNNER_NAME
check("Runner existe", runner.is_file(), str(runner))
check("Runner SHA exacto", sha256(runner) == RUNNER_SHA, sha256(runner))

bundle_manifest = bundle_root / "manifest" / BUNDLE_MANIFEST_NAME
check("Bundle manifest existe", bundle_manifest.is_file(), str(bundle_manifest))
check("Bundle manifest SHA exacto", sha256(bundle_manifest) == BUNDLE_MANIFEST_SHA, sha256(bundle_manifest))

continuity_manifest = (
    bundle_root
    / "results/metrics/phase_g/continuity_series_preparation/v2_2026-08-25"
    / CONTINUITY_MANIFEST_NAME
)
check("Continuity manifest existe", continuity_manifest.is_file(), str(continuity_manifest))
check("Continuity manifest SHA exacto", sha256(continuity_manifest) == CONTINUITY_MANIFEST_SHA, sha256(continuity_manifest))

auth_text = auth074_path.read_text(encoding="utf-8")
check(
    "074 autoriza DATA-SCALING-EXPLORATORY-001",
    "DATA-SCALING-EXPLORATORY-001 = AUTHORIZED FOR EXECUTION" in auth_text,
)
check(
    "074 autoriza runner Phase G",
    "PHASE-G-SCIENTIFIC-RUNNER = AUTHORIZED FOR EXECUTION" in auth_text,
)
check("074 congela runner SHA", RUNNER_SHA in auth_text)
check("074 congela bundle SHA", BUNDLE_SHA in auth_text)

input_dir = bundle_root / "data/processed/ugr16/phase_g"
observed_inputs = sorted(p.name for p in input_dir.iterdir() if p.is_file())
check("Bundle contiene exactamente cinco Parquet científicos", len(observed_inputs) == 5, repr(observed_inputs))

print("PHASE_G_FULL_EXECUTION_CRYPTO_GATE=PASS")


## 4. Ejecución científica productiva

Esta celda ejecuta por primera vez:

```text
DATA-SCALING-EXPLORATORY-001
```

Parámetros operativos:

```text
device = cuda
hard-stage-seconds = 1800
```

`hard-stage-seconds` es únicamente un límite de protección de recursos; no modifica
hiperparámetros ni criterios científicos.

**No interrumpas manualmente la celda mientras avance con normalidad.**


In [ ]:
cmd = [
    sys.executable,
    str(runner),
    "--execute",
    "--project-root", str(bundle_root),
    "--manifest", str(continuity_manifest),
    "--input-dir", str(input_dir),
    "--scientific-out-dir", str(SCIENTIFIC_OUT),
    "--device", "cuda",
    "--hard-stage-seconds", str(HARD_STAGE_SECONDS),
    "--authorization-074", str(auth074_path),
    "--authorization-074-sha", AUTH074_SHA,
]

print("Comando productivo:")
print(" ".join(cmd))
print()
print("=" * 128)
print("INICIO DATA-SCALING-EXPLORATORY-001")
print("=" * 128)

started = time.time()

with EXEC_CONSOLE.open("x", encoding="utf-8", buffering=1) as log:
    proc = subprocess.Popen(
        cmd,
        cwd=bundle_root,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
    return_code = proc.wait()

elapsed_seconds = time.time() - started

print()
print("=" * 128)
print("FIN DEL RUNNER")
print("=" * 128)
print("RUNNER_RETURN_CODE =", return_code)
print("ELAPSED_SECONDS =", elapsed_seconds)
print("EXEC_CONSOLE_SHA256 =", sha256(EXEC_CONSOLE))

execution_state = {
    "return_code": int(return_code),
    "elapsed_seconds": float(elapsed_seconds),
    "console_sha256": sha256(EXEC_CONSOLE),
    "scientific_output_dir_exists": SCIENTIFIC_OUT.exists(),
}

print("No se interpreta todavía el resultado: la siguiente celda audita y empaqueta.")


## 5. Auditoría post-run y empaquetado

Esta celda **no reentrena**. Comprueba el estado publicado por el runner y empaqueta
la evidencia, incluso si el resultado fuese `PARTIAL_FAIL`.

- `RC=0` exige `PASS`, `failed_cells=0` y 124.320 predicciones.
- `RC=2` se conserva como `PARTIAL_FAIL` sin reejecución automática.
- Otro RC con ausencia de outputs se registra como incidente de infraestructura/ejecución.


In [ ]:
scientific_meta = None
scientific_status = "NO_OUTPUT"
scientific_artifacts = {}
post_checks = {}

if SCIENTIFIC_OUT.exists():
    observed = sorted(p.name for p in SCIENTIFIC_OUT.iterdir() if p.is_file())
    check("Set de outputs científicos exacto", observed == sorted(EXPECTED_SCIENTIFIC_FILES), repr(observed))

    scientific_manifest_path = SCIENTIFIC_OUT / "phase_g_data_scaling_manifest_v1_2026-08-25.json"
    scientific_report_path = SCIENTIFIC_OUT / "phase_g_data_scaling_report_v1_2026-08-25.txt"

    scientific_meta = json.loads(scientific_manifest_path.read_text(encoding="utf-8"))
    scientific_status = scientific_meta.get("status")

    check("Campaign ID exacto", scientific_meta.get("campaign_id") == "DATA-SCALING-EXPLORATORY-001", scientific_meta.get("campaign_id"))
    check("Runner SHA exacto en manifest", scientific_meta.get("runner_sha256") == RUNNER_SHA, scientific_meta.get("runner_sha256"))

    for name in EXPECTED_SCIENTIFIC_FILES:
        p = SCIENTIFIC_OUT / name
        scientific_artifacts[name] = {
            "sha256": sha256(p),
            "bytes": p.stat().st_size,
        }

    if return_code == 0:
        check("RC0 implica status PASS", scientific_status == "PASS", scientific_status)
        check("failed_cells = 0", scientific_meta["counts"]["failed_cells"] == 0, scientific_meta["counts"]["failed_cells"])
        check("prediction_rows = 124320", scientific_meta["counts"]["prediction_rows"] == 124320, scientific_meta["counts"]["prediction_rows"])
        check("metric_rows = 64", scientific_meta["counts"]["metric_rows"] == 64, scientific_meta["counts"]["metric_rows"])
        check("scaling_summary_rows = 64", scientific_meta["counts"]["scaling_summary_rows"] == 64, scientific_meta["counts"]["scaling_summary_rows"])
        check("runtime_rows = 40", scientific_meta["counts"]["runtime_rows"] == 40, scientific_meta["counts"]["runtime_rows"])
        check("neural_epoch_rows = 32", scientific_meta["counts"]["neural_epoch_rows"] == 32, scientific_meta["counts"]["neural_epoch_rows"])
        check("cell_status_rows = 40", scientific_meta["counts"]["cell_status_rows"] == 40, scientific_meta["counts"]["cell_status_rows"])
        final_wrapper_status = "PASS"
    elif return_code == 2:
        check("RC2 implica PARTIAL_FAIL", scientific_status == "PARTIAL_FAIL", scientific_status)
        final_wrapper_status = "PARTIAL_FAIL"
    else:
        final_wrapper_status = f"UNEXPECTED_RC_{return_code}_WITH_OUTPUTS"
else:
    if return_code == 0:
        raise RuntimeError("RC=0 pero no existe directorio científico.")
    final_wrapper_status = f"NO_OUTPUT_RC_{return_code}"

report_lines = [
    "=" * 128,
    "PHASE G — FULL EXECUTION COLAB V1R1",
    "=" * 128,
    f"Wrapper status: {final_wrapper_status}",
    f"Runner return code: {return_code}",
    f"Runner elapsed seconds: {elapsed_seconds:.6f}",
    f"Runner SHA-256: {RUNNER_SHA}",
    f"Bundle SHA-256: {BUNDLE_SHA}",
    f"074 SHA-256: {AUTH074_SHA}",
    f"Runtime Python: {runtime['python']}",
    f"Runtime PyTorch: {runtime['torch']}",
    f"Runtime CUDA: {runtime['torch_cuda']}",
    f"Runtime cuDNN: {runtime['cudnn']}",
    f"GPU: {runtime['gpu_name']}",
    f"Hard stage seconds: {HARD_STAGE_SECONDS}",
    f"Execution console SHA-256: {sha256(EXEC_CONSOLE)}",
    f"Scientific output directory exists: {SCIENTIFIC_OUT.exists()}",
    f"Scientific status: {scientific_status}",
]

if scientific_meta is not None:
    report_lines += [
        f"Prediction rows: {scientific_meta['counts']['prediction_rows']}",
        f"Metric rows: {scientific_meta['counts']['metric_rows']}",
        f"Failed cells: {scientific_meta['counts']['failed_cells']}",
    ]

report_lines += [
    "",
    f"DATA-SCALING-EXPLORATORY-001 = {scientific_status}",
    "",
]

WRAPPER_REPORT.write_text("\n".join(report_lines), encoding="utf-8")

members = [
    (EXEC_CONSOLE, f"phase_g_data_scaling_results_v1_2026-08-25/{EXEC_CONSOLE.name}"),
    (auth074_path, f"phase_g_data_scaling_results_v1_2026-08-25/governance/{AUTH074_NAME}"),
]
if SCIENTIFIC_OUT.exists():
    for p in SCIENTIFIC_OUT.iterdir():
        if p.is_file():
            members.append((p, f"phase_g_data_scaling_results_v1_2026-08-25/scientific_outputs/{p.name}"))

deterministic_zip(RESULTS_ZIP, members)

wrapper_manifest_obj = {
    "wrapper_id": WRAPPER_ID,
    "wrapper_version": WRAPPER_VERSION,
    "date_tag": DATE_TAG,
    "status": final_wrapper_status,
    "campaign_id": "DATA-SCALING-EXPLORATORY-001",
    "protocol_id": "PHASE-G-PROTOCOL-001",
    "runner": {
        "name": RUNNER_NAME,
        "sha256": RUNNER_SHA,
        "return_code": int(return_code),
        "hard_stage_seconds": HARD_STAGE_SECONDS,
    },
    "bundle": {
        "name": BUNDLE_NAME,
        "sha256": BUNDLE_SHA,
        "manifest_sha256": BUNDLE_MANIFEST_SHA,
    },
    "authorization_074": {
        "name": AUTH074_NAME,
        "sha256": AUTH074_SHA,
    },
    "runtime": runtime,
    "execution": {
        "elapsed_seconds": float(elapsed_seconds),
        "console_sha256": sha256(EXEC_CONSOLE),
        "scientific_output_dir_exists": SCIENTIFIC_OUT.exists(),
        "scientific_status": scientific_status,
    },
    "scientific_manifest": scientific_meta,
    "scientific_artifacts": scientific_artifacts,
    "results_zip": {
        "name": RESULTS_ZIP.name,
        "sha256": sha256(RESULTS_ZIP),
        "bytes": RESULTS_ZIP.stat().st_size,
    },
    "wrapper_report": {
        "name": WRAPPER_REPORT.name,
        "sha256": sha256(WRAPPER_REPORT),
        "bytes": WRAPPER_REPORT.stat().st_size,
    },
    "governance_assertions": {
        "phase_a_to_f_modified": False,
        "model_reselection": False,
        "retuning": False,
        "best_seed_selection": False,
        "runtime_package_changes": False,
        "raw_tar_access": False,
    },
}

WRAPPER_MANIFEST.write_text(
    json.dumps(wrapper_manifest_obj, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("=" * 128)
print("POST-RUN SUMMARY")
print("=" * 128)
print("WRAPPER_STATUS =", final_wrapper_status)
print("RUNNER_RETURN_CODE =", return_code)
print("SCIENTIFIC_STATUS =", scientific_status)
print("RESULTS_ZIP =", RESULTS_ZIP)
print("RESULTS_ZIP_SHA256 =", sha256(RESULTS_ZIP))
print("WRAPPER_REPORT_SHA256 =", sha256(WRAPPER_REPORT))
print("WRAPPER_MANIFEST_SHA256 =", sha256(WRAPPER_MANIFEST))

if final_wrapper_status == "PASS":
    print("PHASE_G_DATA_SCALING_FULL_EXECUTION=PASS")
elif final_wrapper_status == "PARTIAL_FAIL":
    print("PHASE_G_DATA_SCALING_FULL_EXECUTION=PARTIAL_FAIL / STOP")
else:
    print("PHASE_G_DATA_SCALING_FULL_EXECUTION=FAIL_OR_INCIDENT / STOP")


## 6. Descargar la evidencia

Descarga los tres artefactos y adjúntalos al chat técnico:

```text
phase_g_data_scaling_results_v1_2026-08-25.zip
phase_g_data_scaling_full_execution_colab_v1r1_2026-08-25_report.txt
phase_g_data_scaling_full_execution_colab_v1r1_2026-08-25_manifest.json
```

Si el estado es `PARTIAL_FAIL` o `FAIL_OR_INCIDENT`, **no repitas la campaña** sin
auditoría previa.


In [ ]:
from google.colab import files

files.download(str(RESULTS_ZIP))
files.download(str(WRAPPER_REPORT))
files.download(str(WRAPPER_MANIFEST))

print("=" * 128)
print("DESCARGAS SOLICITADAS")
print("=" * 128)
print("RESULTS ZIP:", RESULTS_ZIP.name, sha256(RESULTS_ZIP))
print("WRAPPER REPORT:", WRAPPER_REPORT.name, sha256(WRAPPER_REPORT))
print("WRAPPER MANIFEST:", WRAPPER_MANIFEST.name, sha256(WRAPPER_MANIFEST))
print("FINAL WRAPPER STATUS:", final_wrapper_status)
print("=" * 128)
